# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `OpenEOJobManagerZonalStats` through the multi-user scheduler. openEO creates sensor-preprocessed physical-band NetCDF cubes per spatial batch. IQR cleaning, null filling, pixel indices, and parcel statistics run locally, producing one GeoParquet row per parcel with dated features and a separate annual summary. See [the detailed workflow](../../../docs/data_preparation/openeo/WORKFLOW.md).

In [ ]:
from pathlib import Path
import os
from IPython.display import display

from data_preparation.parcel_stats import estimate_utm_epsg_from_parcels
from data_preparation.parcel_stats.job_manager import compute_tile_buffer_metres
from data_preparation.parcel_stats.multiuser import (
    load_openeo_users_from_db,
    merge_and_save_results,
    run_parallel_extractions,
)
from shared.io import read_data, write_data
from shared.spatial_statistics import spatial_clustering_using_buffer


## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`optional temporal reduction, or retain acquisitions`
`per-band, per-time-slice raster IQR outlier removal when enabled`
`local null filling`
`local pixel indices from cleaned physical bands`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [ ]:
# ============================================================================
# STEP 1: Read data and re-project
# ============================================================================
# Read the data and re-project to the desired EPSG

WORK_PATH = Path("C:/work_sat")
RUN_IDENTIFIER = "neuro_2024-2025"

# PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "sample_parcels.csv"
# gdf_parcels = read_data(str(PARCELS_PATH))
# print("Initial parcels data:")
# display(gdf_parcels.head())

# print(f"Parcels before dropping duplicate geometries: {len(gdf_parcels)}")
# gdf_parcels = gdf_parcels.drop_duplicates(subset=["geom_wkt"]).copy()
# print(f"Parcels after dropping duplicate geometries: {len(gdf_parcels)}")

# gdf_parcels = gpd.GeoDataFrame(gdf_parcels, geometry=gpd.GeoSeries.from_wkt(gdf_parcels["geom_wkt"]), crs="EPSG:2100")
# print(f"Number of parcels: {len(gdf_parcels)}")
# gdf_parcels = geom_l.return_valid_geometries(gdf_parcels)
# gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
# print(f"Number of parcels after ensuring valid geometries and converting multipolygons to polygons: {len(gdf_parcels)}")
# gdf_parcels = gdf_parcels.drop(columns=["geom_wkt"])

# write_data(gdf_parcels, str(PARCELS_PATH).replace(".csv", ".gpkg"))

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "sample_parcels.gpkg"
gdf_parcels = read_data(str(PARCELS_PATH))
display(gdf_parcels)

In [ ]:
print(gdf_parcels.crs)
WORKING_EPSG = estimate_utm_epsg_from_parcels(gdf_parcels)
WORKING_EPSG

In [ ]:
# ============================================================================
# Define the class column and parcel ID column names.
# These are used in the extraction process to identify the class of each parcel
# and its unique identifier.
# ============================================================================
class_column = "crop_category"
parcel_id_column = "parcel_id"

In [ ]:
gdf_parcels = gdf_parcels[gdf_parcels[class_column] != "jjjj"]
label_counts = gdf_parcels[class_column].value_counts()
print(f"Number of unique labels: {len(label_counts)}")
print(gdf_parcels[class_column].value_counts())
print(50*"-")
rare_labels = label_counts[label_counts < 10].index
gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = "9999"
print(f"Number of unique labels after grouping rare labels: {len(gdf_parcels[class_column].value_counts())}")
print(gdf_parcels[class_column].value_counts())

cond = gdf_parcels[class_column] == "9999"
print(50*"-")
print(f"Excluded parcels with label '9999': {cond.sum()} out of {len(gdf_parcels)}")
gdf_parcels = gdf_parcels[~cond].copy()
print(f"Selected parcels after excluding label '9999': {len(gdf_parcels)}")

In [ ]:
# Reuse saved cluster labels; generate them only when the input has none.
CLUSTER_FIELD = "cluster"
if CLUSTER_FIELD in gdf_parcels.columns:
    gdf_grouped = gdf_parcels.copy()
else:
    gdf_grouped = spatial_clustering_using_buffer(
        gdf_parcels.to_crs(epsg=WORKING_EPSG),
        parcel_id_column,
        min_cluster_distance_in_m=20_000,
        field_cluster=CLUSTER_FIELD,
        plot_results=False,
    )

# groupby would silently omit parcels with missing labels.
if gdf_grouped[CLUSTER_FIELD].isna().any():
    raise ValueError("Every parcel must have a cluster label before partitioning.")

write_data(gdf_grouped, str(PARCELS_PATH).replace(".csv", ".gpkg").replace("sample_parcels", "cleaned_grouped_parcels"))

## Multi-user extraction planning

Accounts are loaded from PostgreSQL using `load_openeo_users_from_db()`. Install the `database` extra and set
`DB_NAME`, `TBL_USERS`, and `POSTGRES_*` in the kernel environment; `.env` is not loaded automatically.
The account table requires non-empty text `username` and `password` columns with unique usernames.
Accounts are sorted by username. Explicitly restore the original account order when resuming legacy jobs
without saved owners or using partition scheduling. See [database account setup](../../../docs/data_preparation/openeo/CREDENTIALS.md).

Create one spatial partition per cluster, then plan all tile batches before
submission. A shared queue lets every account process batches from any partition. With `fit_tiles_to_parcels=True`, a cube covers the
group's complete bounding box plus `TILE_BUFFER_METRES` on every side:

- width = east - west + 2 * buffer
- height = north - south + 2 * buffer

Only if either dimension exceeds `TILE_SIZE_METRES` does the planner split the
group spatially, then fit and check each subgroup in the same way. Parcels stay
whole and belong to exactly one job. A single parcel that exceeds the limit with
its buffer raises an error before submission.

A large partition can use all available accounts at once. Each account stays
within OPENEO_JOBS_PER_USER. Existing cubes are reused; previously submitted
jobs resume on their owning account. Keep the original account order when
first resuming a legacy run, which did not record account names.

Partition outputs and the merge step stay the same. Local statistics run after
the shared download queue completes, using LOCAL_BATCH_WORKERS per active
partition (up to one active partition per account).


In [ ]:
# ============================================================================
# STEP 2: Load accounts for the shared batch queue
# ============================================================================
# Accounts share unsubmitted batches from all cluster partitions.

OPENEO_JOBS_PER_USER = 2  # This is default by openEO
users_list = load_openeo_users_from_db()
print(f"Loaded {len(users_list)} openEO user(s):")
for i, (user, _) in enumerate(users_list, 1):
    print(f"  User {i}: {user}")

In [ ]:
# Keep each cluster together; the existing tile planner handles its size limit.
spatial_parts = [
    part.copy()
    for _, part in gdf_grouped.groupby(CLUSTER_FIELD, sort=True, observed=True)
]
assert sum(len(part) for part in spatial_parts) == len(gdf_grouped)

print(f"Created {len(spatial_parts)} cluster partitions:")
for i, part in enumerate(spatial_parts, 1):
    print(f"  Partition {i}: {part[CLUSTER_FIELD].iloc[0]} - {len(part)} parcels")


In [ ]:
# Maximum requested cube width AND height, including the buffer.
# Each group gets its own fitted extent; only oversized groups are split.
TILE_SIZE_METRES = 20_000
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_grouped, WORKING_EPSG)


In [ ]:
print(f"Maximum buffered cube width/height: {TILE_SIZE_METRES:,} m")
print(f"Buffer on each side: {TILE_BUFFER_METRES:,} m")


In [ ]:
# ============================================================================
# Confirm the planned global remote concurrency.
# ============================================================================
TOTAL_REMOTE_JOB_CAPACITY = len(users_list) * OPENEO_JOBS_PER_USER
print(
    f"Remote capacity: {len(users_list)} users x {OPENEO_JOBS_PER_USER} jobs "
    f"= {TOTAL_REMOTE_JOB_CAPACITY} simultaneous jobs"
)


In [ ]:
# ============================================================================
# STEP 3: Create extraction configuration dictionary
# ============================================================================
# Single dictionary containing all parameters for the extraction process,
# to be passed to each parallel worker.

START_DATE = "2024-10-01"
END_DATE = "2025-09-30"
OUTPUT_DIR = WORK_PATH / RUN_IDENTIFIER / "1_eo_parcel_stats_fill"

extraction_config = {
    # Output paths
    "output_dir": OUTPUT_DIR,
    "run_identifier": RUN_IDENTIFIER,

    # Parcel configuration
    "parcel_id_field": parcel_id_column,

    # Temporal bounds
    "start_date": START_DATE,
    "end_date": END_DATE,

    # Spatial configuration
    "working_epsg": WORKING_EPSG,

    # Fit cluster bounds plus buffer; split only above the maximum size.
    "fit_tiles_to_parcels": True,
    "tile_size_metres": TILE_SIZE_METRES,
    "tile_buffer_metres": TILE_BUFFER_METRES,

    # Parallel processing
    "openeo_parallel_jobs": OPENEO_JOBS_PER_USER,
    "job_poll_seconds": 30,

    # Sentinel-2 configuration
    "sentinel2_bands": ["B02", "B03", "B04", "B05", "B08", "B11", "B12"],
    "sentinel2_indices": ["EVI", "NDMI", "NDRE", "NDVI", "NDWI",  "SAVI"],

    # Sentinel-1 configuration
    "sentinel1_bands": ["VV", "VH"],
    "sentinel1_indices": ["R", "RVI"],
    "sentinel1_orbit_direction": "ASCENDING",  # ASCENDING, DESCENDING, or BOTH

    # Data cleaning (IQR outlier removal)
    "remove_outliers": False,
    "iqr_quantiles": (0.1, 0.9),
    "iqr_multiplier": 1.5,
    "iqr_min_valid_pixels": 20,

    # Null filling
    "fill_nulls": True,
    "temporal_fill_mode": "bidirectional",  # or past_only
    "minimum_parcel_pixels": 3,
    "minimum_observed_fraction_for_fill": 0.20,
    "temporal_fill_window_sizes": (3,),

    # Interpolation
    # 1. Try 3x3, then 5x5, then 7x7; if all fail, leave null.
    "spatial_fill_window_sizes": (3, 5, 7),
    # Fill remaining nulls using the specified interpolation method and maximum distance.
    "interpolation_method": "nearest",  # "idw"
    "interpolation_max_distance_in_meters": 500,

    # Temporal period and reducer for the temporal aggregation of the data.
    # "1M" means monthly aggregation, and "none" means no temporal reduction (keep all observations).
    "temporal_period": "1M",
    "temporal_reducer": "none",  # All valid pixels will be used

    # Statistics to calculate for the selected bands and indices; these are calculated for each parcel and each temporal period.
    "spatial_statistics": ["mean", "median", "sd", "min", "max", "range"],

    # Each worker loads one complete monthly cube; use one unless RAM has been sized for parallel cubes.
    "batch_workers": 4,  # For local processes only

    "remove_nc_after_completion": True,
    "resume_completed_partitions": True,
}

In [ ]:
# ============================================================================
# STEP 4: Plan all batches, then download across all users
# ============================================================================
# Plan all partition tiles first; any free account slot takes the next batch.
# Batches from the same partition may run on different accounts.
run_parallel_extractions(
    spatial_parts=spatial_parts,
    users_list=users_list,
    config=extraction_config,
    scheduling="batches",
)

# Each partition result is saved below OUTPUT_DIR/partition_<n>.
print("\nExtraction complete!")

In [ ]:
# ============================================================================
# STEP 4: Merge results and save to files
# ============================================================================
# Combines all partition results and removes duplicates
# Saves one file:
#   1. satellite_parcel_time_stats.geoparquet (full results with geometry)

merged_gdf, geoparquet_path = merge_and_save_results(
    output_dir=OUTPUT_DIR,
    parcel_id_column=parcel_id_column,
    working_epsg=WORKING_EPSG
)

In [ ]:
# ============================================================================
# STEP 5: Inspect results
# ============================================================================

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")

print("\nMerged GeoDataFrame:")
print(f"  Rows: {len(merged_gdf)}")
print(f"  Columns: {len(merged_gdf.columns)}")
print(f"  Has geometry: {merged_gdf.geometry.name in merged_gdf.columns}")

print("\nOutput files:")
print(f"  GeoParquet: {geoparquet_path}")
print(f"    Size: {geoparquet_path.stat().st_size / 1e6:.1f} MB")

print("\nFirst 5 rows:")
display(merged_gdf.head())